# Vineyard Rows — YOLO11-seg training (Colab T4)

Fine-tunes YOLO11-seg on the `row` class using patches sliced from the 2 fully-annotated Moldova example tiles. The Riseholme-pretrained model failed on our domain (1/25 rows); this trains on our exact imagery.

**Steps:**
1. Runtime → Change runtime type → **GPU (T4)**.
2. Run every cell top-to-bottom.
3. Upload `rows_yolo.zip` when prompted (cell 2).
4. Download `best.pt` at the end and drop it into `pipeline/runs/segment/riseholme/weights/`.

In [ ]:
# 1. Env check + install
!nvidia-smi -L
!pip -q install ultralytics==8.3.0
import torch; print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
# 2. Upload rows_yolo.zip (built locally by pipeline/build_rows_dataset.py)
from google.colab import files
up = files.upload()          # pick rows_yolo.zip
!rm -rf rows_yolo && unzip -q rows_yolo.zip
!ls rows_yolo && echo '---' && cat rows_yolo/data.yaml

In [ ]:
# 3. Fix the dataset path for Colab (data.yaml has a local absolute path)
import os
cwd = os.getcwd()
yaml_path = 'rows_yolo/data.yaml'
open(yaml_path, 'w').write(
    f"path: {cwd}/rows_yolo\n"
    "train: images/train\n"
    "val: images/val\n"
    "nc: 1\n"
    "names: ['row']\n"
)
print(open(yaml_path).read())

In [ ]:
# 4. Train. Small dataset (2 source tiles) -> lean on heavy augmentation.
from ultralytics import YOLO

model = YOLO('yolo11s-seg.pt')   # s > n: only 2 source tiles, we want capacity
model.train(
    data='rows_yolo/data.yaml',
    epochs=150,
    imgsz=640,
    batch=16,               # T4 handles this at 640
    device=0,
    patience=40,
    project='runs', name='rows',
    # augmentation: the whole point — 2 tiles must generalize
    degrees=180, fliplr=0.5, flipud=0.5, scale=0.5,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, mosaic=1.0, close_mosaic=15,
    plots=True,
)
print('best:', 'runs/rows/weights/best.pt')

In [ ]:
# 5. Quick sanity: predict on a val patch and show masks found
from ultralytics import YOLO
import glob
m = YOLO('runs/rows/weights/best.pt')
val = sorted(glob.glob('rows_yolo/images/val/*.png'))[:4]
for f in val:
    r = m.predict(f, imgsz=640, conf=0.25, verbose=False)[0]
    n = 0 if r.masks is None else len(r.masks)
    print(f.split('/')[-1], '-> row masks:', n)

In [ ]:
# 6. Download the trained weights
from google.colab import files
files.download('runs/rows/weights/best.pt')